In [5]:
import pandas as pd
from scipy.special import comb
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest
import ast
import math
import numpy as np

from win_loss_utils import c1, c2, c3, split

In [ ]:
soccer = pd.read_csv("../../data/processed/soccer_binary.csv")
us = pd.read_csv("../../data/processed/us_leagues.csv")

# Get list equivalent instead of string

soccer['Sequence_Numeric'] = soccer['Sequence'].apply(ast.literal_eval)
us['Sequence_Numeric'] = us['Sequence'].apply(ast.literal_eval)

In [6]:
## Add Columns n, n1, n2, k, k1, k2

# total length of each sequence
us['n'] = us['Sequence_Numeric'].apply(len)

# split once per row
splits = us['Sequence_Numeric'].apply(split)

# lengths
us['n1'] = splits.apply(lambda x: len(x[0]))
us['n2'] = splits.apply(lambda x: len(x[1]))

# sums
us['k'] = us['Sequence_Numeric'].apply(sum)
us['k1'] = splits.apply(lambda x: sum(x[0]))
us['k2'] = splits.apply(lambda x: sum(x[1]))

# ---- #

soccer['n'] = soccer['Sequence_Numeric'].apply(len)

splits = soccer['Sequence_Numeric'].apply(split)

soccer['n1'] = splits.apply(lambda x: len(x[0]))
soccer['n2'] = splits.apply(lambda x: len(x[1]))

soccer['k'] = soccer['Sequence_Numeric'].apply(sum)
soccer['k1'] = splits.apply(lambda x: sum(x[0]))
soccer['k2'] = splits.apply(lambda x: sum(x[1]))

In [7]:
# Apply C1

soccer['c1'] = [c1(seq) for seq in soccer['Sequence_Numeric']]
us['c1'] = [c1(seq) for seq in us['Sequence_Numeric']]

In [8]:
# Apply C2

soccer['c2'] = [c2(seq) for seq in soccer['Sequence_Numeric']]
us['c2'] = [c2(seq) for seq in us['Sequence_Numeric']]

C:\Users\ryanj\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\stats\weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std


In [9]:
N = 1000

soccer['c3'] = [c3(seq, N) for seq in soccer['Sequence_Numeric']]
us['c3'] = [c3(seq, N) for seq in us['Sequence_Numeric']]

In [10]:
## Recalculate top 100 in c3 column 

us_indexes = list(us.sort_values('c3', ascending=False).head(100).index)
soccer_indexes = list(soccer.sort_values('c3', ascending = False).head(100).index)

N = 100000

mask = us.index.isin(us_indexes)
us.loc[mask, 'c3'] = us.loc[mask, 'Sequence_Numeric'].apply(lambda seq: c3(seq, N))

mask = soccer.index.isin(soccer_indexes)
soccer.loc[mask, 'c3'] = soccer.loc[mask, 'Sequence_Numeric'].apply(lambda seq: c3(seq, N))

In [ ]:
us_out = us[['League', 'Season', 'Team', 'Sequence', 'n', 'n1', 'n2', 'k', 'k1', 'k2', 'c1', 'c2', 'c3']]
soccer_out = soccer[['League', 'Season', 'Team', 'Sequence', 'n', 'n1', 'n2', 'k', 'k1', 'k2', 'c1', 'c2', 'c3']]

soccer_output = "../../output/csvs/soccer_second_half_cscores.csv"
us_output = "../../output/csvs/us_leagues_second_half_cscores.csv"

soccer_out.to_csv(soccer_output, index = False)
us_out.to_csv(us_output, index = False)